In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/src/rewritten/o4_mini_high/checkpoints/post_cell_6.pickle

[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba924fd9f0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba93f732b0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba93f732e0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba93f73430>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba924fe9b0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba924fe9e0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba924fea10>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba924adc30>]]
trying: ['df']
me:  8
trying: ['sns']
me:  0
trying: ['pd']
me:  0
trying: ['factor']
me:  4
trying: ['time']
me:  0
trying: ['plt']
me:  0
trying: ['np']
me:  0
trying: ['passmark']
me:  1
Declaring variable sns
Declaring variable pd
Declaring variable time
Declaring variable plt
Declaring variable np
Declaring variable pass

In [4]:
%%RecordEvent
%%time
### cell 7 ###

### cell 7 – optimized for cudf (no np.where, no CPU round-trip)
# ensure the column is numeric on the GPU
df["reading score"] = pd.to_numeric(df["reading score"], errors="coerce")

# initialize the pass status to 'P' entirely on the GPU
# then mask all rows with reading score < passmark to 'F'
df["Reading_PassStatus"] = 'P'
df["Reading_PassStatus"] = df["Reading_PassStatus"].mask(
    df["reading score"] < passmark,
    'F'
)

# value_counts stays on the GPU
df["Reading_PassStatus"].value_counts()

CPU times: user 40.1 ms, sys: 43.3 ms, total: 83.4 ms
Wall time: 134 ms


Reading_PassStatus
P    974000
F     26000
Name: count, dtype: int64

In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/src/rewritten/o4_mini_high/checkpoints/post_cell_7.pickle

migration speed (bps): 757070462.7616774
---------------------------
variables to migrate:
time 72
plt 72
sns 72
np 72
passmark 28
pd 72
df 48
factor 28
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/src/rewritten/o4_mini_high/checkpoints/post_cell_7.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 2 =======
Input variables ['df']
Active variables ['df']
Intermediate variables ['factor']
Future variables []
Modified dataframes
======= Cell 3 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 4 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 5 =======
Input variables ['df']
Active variables []
Intermediate variables []
Future variables ['df']
Modified dataframes
======= Cell 6 =======
Input variables ['passmark', 'df']
Active variables ['df']
Intermediate variables []
Future variables []
Modified dataframes
  df
    Input columns: set()
  

In [7]:

with open("/home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/src/opt_cell_exec_info_7_try_1.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[7], f)


In [8]:
opt_output = Out.get(4)

In [9]:
df_opt = df
%LoadCheckpoint /home/dias-benchmarks/notebooks/spscientist/student-performance-in-exams/src/annotated/checkpoints/post_cell_7.pickle
assert compare_df(df_opt, df)

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


[[<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9232f4f0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9232f6d0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9232f490>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9232f460>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9239a920>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9239b100>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9239bcd0>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9239bb80>], [<elastic.core.graph.variable_snapshot.VariableSnapshot object at 0x7fba9239bc40>]]
trying: ['factor']
me:  4
trying: ['plt']
me:  0
trying: ['time']
me:  0
trying: ['pd']
me:  0
trying: ['sns']
me:  0
trying: ['orig_output']
me:  11
trying: ['np']
me:  0
trying: ['df']
me:  10
trying: ['passmark']
me:  1


Declaring variable plt
Declaring variable time
Declaring variable pd
Declaring variable sns
Declaring variable np
Declaring variable passmark
Declaring variable factor
Declaring variable df
Declaring variable orig_output
